In [ ]:
# ===== Single-run GRU training (BEST + LAST ckpt) + (optional) AUGMENTATION =====
# KEY CHANGE: uses the COMPETITION METRIC (weighted Pearson) as the loss:
#   loss = -(pearson_y1 + pearson_y2)/2
#
# Paste + run. Edit the RUN CONFIG block only.

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------------
# Paths
# -------------------------
TRAIN_FILE = Path("/kaggle/input/lob-data/train.parquet")
VALID_FILE = Path("/kaggle/input/lob-data/valid.parquet")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# RUN CONFIG (edit these)
# -------------------------
SEED = 123

# Choose ONE preset (uncomment)
# --- Preset A: GRU_h128_L4_d0.03 ---
HIDDEN = 128
NUM_LAYERS = 4
DROPOUT = 0.03

# --- Preset B: GRU_h64_L4_d0.05 ---
# HIDDEN = 64
# NUM_LAYERS = 4
# DROPOUT = 0.05

LR = 3e-4
WEIGHT_DECAY = 1e-5

# -------------------------
# Model fixed
# -------------------------
INPUT_DIM = 32
D_OUT = 2

# -------------------------
# Training fixed
# -------------------------
BATCH_SIZE = 32
EPOCHS = 60
NUM_WORKERS = 2
PATIENCE = 6
CLIP_NORM = 1.0

# -------------------------
# Augmentation (TRAIN only)
# -------------------------
# Start with AUGMENT=False until you confirm the metric-loss works and improves LB vs val.
AUGMENT = False
DO_VAR_NORM = True
SCALE_LOW, SCALE_HIGH = 0.9, 1.1
NOISE_FRAC = 0.01
EPS = 1e-6

# -------------------------
# Repro
# -------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(SEED)

def _worker_init_fn(worker_id: int):
    s = SEED + worker_id * 1000
    np.random.seed(s)
    random.seed(s)

# -------------------------
# Dataset
# -------------------------
class SeqDataset(Dataset):
    def __init__(self, df: pd.DataFrame, augment: bool = False, global_feat_std: np.ndarray | None = None):
        self.augment = augment
        self.groups = []

        for _, g in df.groupby("seq_ix", sort=False):
            g = g.sort_values("step_in_seq")
            x = g.iloc[:, 3:35].to_numpy(dtype=np.float32)   # 32 features
            y = g.iloc[:, 35:37].to_numpy(dtype=np.float32)  # 2 targets
            n = g["need_prediction"].to_numpy(dtype=np.uint8)
            self.groups.append((x, y, n))

        # compute global feature std (median over sequences), unless provided
        if global_feat_std is None:
            stds = []
            for x, _, _ in self.groups:
                stds.append(x.std(axis=0, ddof=0))
            stds = np.stack(stds, axis=0)
            self.global_feat_std = np.median(stds, axis=0).astype(np.float32)
        else:
            self.global_feat_std = global_feat_std.astype(np.float32)

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx: int):
        x, y, n = self.groups[idx]

        if self.augment and AUGMENT:
            x = x.copy()

            # (1) variance normalisation (per-feature)
            if DO_VAR_NORM:
                seq_std = x.std(axis=0, ddof=0)
                scale_to_target = self.global_feat_std / (seq_std + EPS)
                x *= scale_to_target[None, :]

            # (2) random global scaling
            s = np.random.uniform(SCALE_LOW, SCALE_HIGH)
            x *= np.float32(s)

            # (3) gaussian noise proportional to feature std
            noise_std = (NOISE_FRAC * self.global_feat_std)[None, :]
            x += (np.random.normal(0.0, 1.0, size=x.shape).astype(np.float32) * noise_std)

        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(n)

def collate_stack(batch):
    xs, ys, ns = zip(*batch)
    return torch.stack(xs, 0), torch.stack(ys, 0), torch.stack(ns, 0)

# -------------------------
# Model
# -------------------------
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden, num_layers, dropout, d_out):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            dropout=dropout if num_layers >= 2 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden, d_out)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out)

# -------------------------
# Metric-as-loss (weighted Pearson), torch version
# -------------------------
@torch.no_grad()
def _safe_std_w(x, w, eps=1e-6):
    # x, w are 1D
    mean = (w * x).sum() / (w.sum() + eps)
    var = (w * (x - mean) ** 2).sum() / (w.sum() + eps)
    return torch.sqrt(var + eps), mean

def weighted_pearson_1d(x: torch.Tensor, y: torch.Tensor, w: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """
    x, y, w: 1D tensors of same length
    returns scalar correlation in [-1,1]
    """
    wsum = w.sum() + eps
    mx = (w * x).sum() / wsum
    my = (w * y).sum() / wsum

    xc = x - mx
    yc = y - my

    cov = (w * xc * yc).sum() / wsum
    vx = (w * xc * xc).sum() / wsum
    vy = (w * yc * yc).sum() / wsum

    corr = cov / (torch.sqrt(vx * vy) + eps)
    return corr

def metric_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """
    pred, target: (N, 2) after masking
    loss = -(corr_dim0 + corr_dim1)/2
    weights: mimic your previous weighting: abs(target) clamped
    """
    # weights per dimension
    w0 = target[:, 0].abs().clamp_min(1e-3)
    w1 = target[:, 1].abs().clamp_min(1e-3)

    c0 = weighted_pearson_1d(pred[:, 0], target[:, 0], w0, eps=eps)
    c1 = weighted_pearson_1d(pred[:, 1], target[:, 1], w1, eps=eps)

    # maximise correlation -> minimise negative correlation
    return -0.5 * (c0 + c1)

# -------------------------
# Load data once
# -------------------------
print("Loading train/valid...")
train_df = pd.read_parquet(TRAIN_FILE)
valid_df = pd.read_parquet(VALID_FILE)

train_ds = SeqDataset(train_df, augment=True, global_feat_std=None)
valid_ds = SeqDataset(valid_df, augment=False, global_feat_std=train_ds.global_feat_std)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    collate_fn=collate_stack,
    worker_init_fn=_worker_init_fn if NUM_WORKERS > 0 else None,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    collate_fn=collate_stack,
)

print("DEVICE:", DEVICE, "| gpu:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu")
print(f"CONFIG: hidden={HIDDEN} layers={NUM_LAYERS} dropout={DROPOUT} lr={LR} wd={WEIGHT_DECAY} seed={SEED}")
print(f"AUG: {AUGMENT} | var_norm={DO_VAR_NORM} | scale=[{SCALE_LOW},{SCALE_HIGH}] | noise_frac={NOISE_FRAC}")

# -------------------------
# Train one run
# -------------------------
tag = f"h{HIDDEN}_L{NUM_LAYERS}_do{DROPOUT}_lr{LR:g}_wd{WEIGHT_DECAY:g}_bs{BATCH_SIZE}_seed{SEED}_metricloss1_aug{int(AUGMENT)}"
best_path = OUT_DIR / f"gru_best_{tag}.pt"
last_path = OUT_DIR / f"gru_last_{tag}.pt"

model = GRUModel(INPUT_DIM, HIDDEN, NUM_LAYERS, DROPOUT, D_OUT).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_valid_loss = float("inf")
best_valid_metric = -float("inf")
bad = 0
last_valid_loss = None
last_valid_metric = None

for epoch in range(EPOCHS):
    # ---- train ----
    model.train()
    train_sum = 0.0
    train_batches = 0

    for X, Y, N in train_loader:
        X = X.to(DEVICE, non_blocking=True)
        Y = Y.to(DEVICE, non_blocking=True)
        N = N.to(DEVICE, non_blocking=True)

        pred = model(X)
        mask = (N == 1)
        if mask.sum().item() == 0:
            continue

        p = pred[mask]
        t = Y[mask]

        loss = metric_loss(p, t)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        if CLIP_NORM and CLIP_NORM > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        opt.step()

        train_sum += loss.item()
        train_batches += 1

    train_avg = train_sum / max(1, train_batches)

    # ---- valid ----
    model.eval()
    with torch.no_grad():
        valid_sum = 0.0
        valid_batches = 0
        metric_sum = 0.0

        for X, Y, N in valid_loader:
            X = X.to(DEVICE, non_blocking=True)
            Y = Y.to(DEVICE, non_blocking=True)
            N = N.to(DEVICE, non_blocking=True)

            pred = model(X)
            mask = (N == 1)
            if mask.sum().item() == 0:
                continue

            p = pred[mask]
            t = Y[mask]

            vloss = metric_loss(p, t)   # negative correlation
            valid_sum += vloss.item()

            # report the positive metric too (so you can read it like leaderboard)
            # metric = -loss on that batch (not exactly global, but good indicator)
            metric_sum += (-vloss).item()

            valid_batches += 1

    valid_avg_loss = valid_sum / max(1, valid_batches)
    valid_avg_metric = metric_sum / max(1, valid_batches)

    last_valid_loss = valid_avg_loss
    last_valid_metric = valid_avg_metric

    # checkpoint by VALID LOSS (lower is better)
    if valid_avg_loss < best_valid_loss:
        best_valid_loss = valid_avg_loss
        best_valid_metric = valid_avg_metric
        bad = 0

        torch.save(
            {
                "state_dict": model.state_dict(),
                "input_dim": INPUT_DIM,
                "hidden": HIDDEN,
                "num_layers": NUM_LAYERS,
                "dropout": float(DROPOUT),
                "d_out": D_OUT,
                "epoch": epoch + 1,
                "best_valid_loss": float(best_valid_loss),
                "best_valid_metric": float(best_valid_metric),
                "tag": tag,
                "seed": int(SEED),
                "lr": float(LR),
                "weight_decay": float(WEIGHT_DECAY),
                "augment": bool(AUGMENT),
                "var_norm": bool(DO_VAR_NORM),
                "scale_low": float(SCALE_LOW),
                "scale_high": float(SCALE_HIGH),
                "noise_frac": float(NOISE_FRAC),
                "loss_name": "metric_loss_weighted_pearson",
            },
            best_path,
        )
    else:
        bad += 1

    print(
        f"[{tag}] ep {epoch+1:02d}: "
        f"train_loss={train_avg:.4f} "
        f"valid_loss={valid_avg_loss:.4f} "
        f"valid_metric~={valid_avg_metric:.4f} "
        f"best_loss={best_valid_loss:.4f} bad={bad}/{PATIENCE}"
    )

    if bad >= PATIENCE:
        break

# ---- save last ----
torch.save(
    {
        "state_dict": model.state_dict(),
        "input_dim": INPUT_DIM,
        "hidden": HIDDEN,
        "num_layers": NUM_LAYERS,
        "dropout": float(DROPOUT),
        "d_out": D_OUT,
        "epoch": epoch + 1,
        "valid_loss_last": float(last_valid_loss) if last_valid_loss is not None else None,
        "valid_metric_last": float(last_valid_metric) if last_valid_metric is not None else None,
        "tag": tag,
        "seed": int(SEED),
        "lr": float(LR),
        "weight_decay": float(WEIGHT_DECAY),
        "augment": bool(AUGMENT),
        "var_norm": bool(DO_VAR_NORM),
        "scale_low": float(SCALE_LOW),
        "scale_high": float(SCALE_HIGH),
        "noise_frac": float(NOISE_FRAC),
        "loss_name": "metric_loss_weighted_pearson",
    },
    last_path,
)

print("\nSaved:")
print(" BEST:", best_path)
print(" LAST:", last_path)
print("Best valid loss (negative corr):", best_valid_loss)
print("Best valid metric approx (corr):", best_valid_metric)

Check path for utils.py

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

# Add the Kaggle "Model" input folder that contains utils.py
sys.path.append("/kaggle/input/gru2/other/default/1")

from utils import DataPoint, ScorerStepByStep

DEVICE = "cpu"  # submission is CPU, scoring on CPU is fine

class GRUModel(nn.Module):
    def __init__(self, input_dim=32, hidden=128, num_layers=4, dropout=0.1, d_out=2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            dropout=dropout if num_layers >= 2 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden, d_out)

    def forward(self, x, h=None):
        out, h = self.gru(x, h)
        y = self.head(out)
        return y, h

class PredictionModel:
    def __init__(self, ckpt_path: str):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)

        self.model = GRUModel(
            input_dim=int(ckpt.get("input_dim", 32)),
            hidden=int(ckpt.get("hidden", 128)),
            num_layers=int(ckpt.get("num_layers", 4)),
            dropout=float(ckpt.get("dropout", 0.1)),
            d_out=2,
        ).to(DEVICE)

        self.model.load_state_dict(ckpt["state_dict"])
        self.model.eval()

        self.current_seq_ix = None
        self.h = None

    def _reset(self, seq_ix: int):
        self.current_seq_ix = seq_ix
        self.h = None

    @torch.no_grad()
    def predict(self, data_point: DataPoint):
        if self.current_seq_ix != data_point.seq_ix:
            self._reset(data_point.seq_ix)

        x = torch.from_numpy(data_point.state.astype(np.float32, copy=False)).to(DEVICE).view(1, 1, -1)
        y, self.h = self.model(x, self.h)
        pred = y[0, 0].cpu().numpy().astype(np.float32)

        if not data_point.need_prediction:
            return None

        return np.clip(pred, -6.0, 6.0).astype(np.float32)

# pick BEST checkpoint automatically
best = sorted(Path("/kaggle/working").glob("gru_best_*.pt"))
last = sorted(Path("/kaggle/working").glob("gru_last_*.pt"))

ckpt_path = str(best[-1] if best else last[-1])
print("Using ckpt:", ckpt_path)

test_file = "/kaggle/input/lob-data/valid.parquet"

model = PredictionModel(ckpt_path)
scorer = ScorerStepByStep(test_file)

print("Scoring...")
results = scorer.score(model)
print("Results:", results)